# train_cloud_model

Этот notebook обучает **cloud-модель** на базе нового `preprocess.py`.

Логика пайплайна:
1. Загружаем **base processed dataset**
2. Формируем `X, y` через `build_cloud_dataset()`
3. Делаем `train_test_split`
4. Строим `normalization_config` **только по train**
5. Нормализуем `train` и `test` одинаковым config
6. Применяем **SMOTE только к train**
7. Обучаем `XGBoost`
8. Считаем метрики
9. Сохраняем модель, `normalization_config`, test set и predictions


In [1]:
import json
from pathlib import Path

import joblib
import pandas as pd

from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.model_selection import train_test_split

from xgboost import XGBClassifier

from preprocess import (
    CLOUD_FEATURES,
    build_cloud_dataset,
    build_normalization_config_from_df,
    get_processed_dataframe,
    normalize_features,
    save_feature_list,
    save_json,
)


## 1. Пути и конфиг

In [2]:
ARTIFACTS_DIR = Path("training/artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ARTIFACTS_DIR / "cloud_model.json"
FEATURES_PATH = ARTIFACTS_DIR / "cloud_feature_list.json"
METRICS_PATH = ARTIFACTS_DIR / "cloud_metrics.json"
NORMALIZATION_CONFIG_PATH = ARTIFACTS_DIR / "cloud_normalization_config.json"

TEST_CSV_PATH = ARTIFACTS_DIR / "cloud_test_processed.csv"
TEST_PREDICTIONS_PATH = ARTIFACTS_DIR / "cloud_test_predictions.csv"

RANDOM_STATE = 42
TEST_SIZE = 0.2

SMOTE_ENABLED = True
SMOTE_K_NEIGHBORS = 5


## 2. Загрузка base processed dataset

In [3]:
df = get_processed_dataframe()

print("Base processed dataframe shape:", df.shape)
df.head()


Base processed dataframe shape: (10000, 13)


,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,temp_diff,power_kw
0,298.1,308.6,1551,42.8,0,0,0,0,0,0,0,10.5,6.951591
1,298.2,308.7,1408,46.3,3,0,0,0,0,0,0,10.5,6.826723
2,298.1,308.5,1498,49.4,5,0,0,0,0,0,0,10.4,7.749388
3,298.2,308.6,1433,39.5,7,0,0,0,0,0,0,10.4,5.927505
4,298.2,308.7,1408,40.0,9,0,0,0,0,0,0,10.5,5.897817


## 3. Формируем признаки и target

In [4]:
X, y = build_cloud_dataset(df)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("\nClass distribution:")
print(y.value_counts())

X.head()


Feature matrix shape: (10000, 6)
Target shape: (10000,)

Class distribution:
Machine failure
0    9661
1     339
Name: count, dtype: int64


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min]
0,298.1,10.5,1551,42.8,6.951591,0
1,298.2,10.5,1408,46.3,6.826723,3
2,298.1,10.4,1498,49.4,7.749388,5
3,298.2,10.4,1433,39.5,5.927505,7
4,298.2,10.5,1408,40.0,5.897817,9


## 4. Train / test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("\nTrain class distribution:")
print(y_train.value_counts())
print("\nTest class distribution:")
print(y_test.value_counts())


Train shape: (8000, 6)
Test shape: (2000, 6)

Train class distribution:
Machine failure
0    7729
1     271
Name: count, dtype: int64

Test class distribution:
Machine failure
0    1932
1      68
Name: count, dtype: int64


## 5. Нормализация

`normalization_config` строится **только по train**.

Потом тот же config используется для:
- `X_train`
- `X_test`
- simulator / inference


In [6]:
normalization_config = build_normalization_config_from_df(X_train)

X_train_norm = normalize_features(X_train.copy(), normalization_config)
X_test_norm = normalize_features(X_test.copy(), normalization_config)

print("Normalization config:")
normalization_config


Normalization config:


{'Air temperature [K]': {'mode': 'minmax', 'min': 297.1, 'max': 303.5},
 'temp_diff': {'mode': 'minmax',
  'min': 8.399999999999977,
  'max': 11.400000000000034},
 'Rotational speed [rpm]': {'mode': 'ratio', 'max': 1869.0},
 'Torque [Nm]': {'mode': 'ratio', 'max': 56.3},
 'power_kw': {'mode': 'ratio', 'max': 8.044595673835941},
 'Tool wear [min]': {'mode': 'ratio', 'max': 206.0}}

## 6. Сохраняем test set для cloud evaluation / simulator

In [7]:
test_df = X_test_norm.copy()
test_df["Machine failure"] = y_test.values

test_df.to_csv(TEST_CSV_PATH, index=False)

print("Saved normalized test set to:", TEST_CSV_PATH)
test_df.head()


Saved normalized test set to: training\artifacts\cloud_test_processed.csv


,Air temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_kw,Tool wear [min],Machine failure
2997,0.53125,0.300000,0.719636,1.000000,1.000000,0.742718,0
4871,1.00000,0.100000,0.809524,0.712256,0.789783,0.655340,0
3858,0.84375,0.166667,0.834136,0.667851,0.763059,1.000000,0
951,0.00000,0.766667,0.807384,0.635879,0.703229,0.291262,0
6463,0.53125,0.366667,0.726592,1.000000,1.000000,0.495146,0


## 7. Балансировка train через SMOTE

SMOTE применяется **только к train**.

`X_test` и simulator data **не балансируются**.


In [8]:
if SMOTE_ENABLED:
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=SMOTE_K_NEIGHBORS)
    X_train_final, y_train_final = smote.fit_resample(X_train_norm, y_train)
    print("SMOTE applied successfully")
else:
    X_train_final, y_train_final = X_train_norm.copy(), y_train.copy()
    print("SMOTE disabled")

print("Final train shape:", X_train_final.shape)
print("\nBalanced class distribution:")
print(pd.Series(y_train_final).value_counts())


SMOTE applied successfully
Final train shape: (15458, 6)

Balanced class distribution:
Machine failure
0    7729
1    7729
Name: count, dtype: int64


In [9]:
def sanitize_feature_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [
        col.replace("[", "")
           .replace("]", "")
           .replace("<", "")
           .replace(" ", "_")
        for col in df.columns
    ]
    return df

In [10]:
X_train_final = sanitize_feature_names(X_train_final)
X_test_norm = sanitize_feature_names(X_test_norm)

## 8. Обучение cloud-модели

Для cloud слоя используем `XGBoost`, так как он обычно дает более гибкое и точное моделирование сложных нелинейных зависимостей.


In [11]:
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train_final, y_train_final)

print("Cloud model trained successfully")


Cloud model trained successfully


## 9. Оценка модели

In [12]:
y_pred = model.predict(X_test_norm)
y_prob = model.predict_proba(X_test_norm)[:, 1]

metrics = {
    "model_type": "cloud",
    "model_name": "XGBClassifier",
    "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
    "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
    "recall": round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
    "f1_score": round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
    "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
}

print("Metrics:")
metrics


Metrics:


{'model_type': 'cloud',
 'model_name': 'XGBClassifier',
 'accuracy': 0.953,
 'precision': 0.4058,
 'recall': 0.8235,
 'f1_score': 0.5437,
 'roc_auc': 0.964}

In [13]:
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))


Confusion matrix:
[[1850   82]
 [  12   56]]

Classification report:
              precision    recall  f1-score   support

           0       0.99      0.96      0.98      1932
           1       0.41      0.82      0.54        68

    accuracy                           0.95      2000
   macro avg       0.70      0.89      0.76      2000
weighted avg       0.97      0.95      0.96      2000



## 10. Предсказания на test set

In [14]:
test_predictions_df = X_test_norm.copy()
test_predictions_df["Machine failure"] = y_test.values
test_predictions_df["predicted_label"] = y_pred
test_predictions_df["predicted_probability"] = y_prob

test_predictions_df.to_csv(TEST_PREDICTIONS_PATH, index=False)

print("Saved test predictions to:", TEST_PREDICTIONS_PATH)
test_predictions_df.head()


Saved test predictions to: training\artifacts\cloud_test_predictions.csv


,Air_temperature_K,temp_diff,Rotational_speed_rpm,Torque_Nm,power_kw,Tool_wear_min,Machine failure,predicted_label,predicted_probability
2997,0.53125,0.300000,0.719636,1.000000,1.000000,0.742718,0,1,0.629821
4871,1.00000,0.100000,0.809524,0.712256,0.789783,0.655340,0,0,0.001563
3858,0.84375,0.166667,0.834136,0.667851,0.763059,1.000000,0,1,0.659671
951,0.00000,0.766667,0.807384,0.635879,0.703229,0.291262,0,0,0.002993
6463,0.53125,0.366667,0.726592,1.000000,1.000000,0.495146,0,0,0.352144


## 11. Важность признаков

In [15]:
feature_importance = pd.DataFrame({
    "feature": X_train_norm.columns,
    "importance": model.feature_importances_,
}).sort_values(by="importance", ascending=False)

feature_importance


,feature,importance
2,Rotational speed [rpm],0.312264
4,power_kw,0.215808
3,Torque [Nm],0.172337
5,Tool wear [min],0.169761
1,temp_diff,0.083780
0,Air temperature [K],0.046050


## 12. Сохранение артефактов

In [16]:
model.get_booster().save_model(str(MODEL_PATH))
save_feature_list(list(CLOUD_FEATURES), FEATURES_PATH)
save_json(normalization_config, NORMALIZATION_CONFIG_PATH)

with open(METRICS_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print("Saved model to:", MODEL_PATH)
print("Saved features to:", FEATURES_PATH)
print("Saved metrics to:", METRICS_PATH)
print("Saved normalization config to:", NORMALIZATION_CONFIG_PATH)


Feature list saved to: training\artifacts\cloud_feature_list.json
Saved model to: training\artifacts\cloud_model.json
Saved features to: training\artifacts\cloud_feature_list.json
Saved metrics to: training\artifacts\cloud_metrics.json
Saved normalization config to: training\artifacts\cloud_normalization_config.json


## 13. Пример одного предсказания

In [17]:
sample_input = X_test_norm.iloc[[0]].copy()

sample_risk = float(model.predict_proba(sample_input)[0, 1])
sample_pred = int(model.predict(sample_input)[0])

result = {
    "risk_score": round(sample_risk, 4),
    "prediction": sample_pred,
    "prediction_label": "HIGH_RISK" if sample_pred == 1 else "NORMAL",
    "model_name": "cloud_model.json",
    "features": sample_input.to_dict(orient="records")[0],
}

result


{'risk_score': 0.6298,
 'prediction': 1,
 'prediction_label': 'HIGH_RISK',
 'model_name': 'cloud_model.json',
 'features': {'Air_temperature_K': 0.5312499999999983,
  'temp_diff': 0.3000000000000057,
  'Rotational_speed_rpm': 0.7196361690743713,
  'Torque_Nm': 1.0,
  'power_kw': 1.0,
  'Tool_wear_min': 0.7427184466019418}}